# Benchmark: Drive vs disco local para reannotación HaMeR

Descarga S2, extrae 200 frames, mide fps del pipeline completo
(MediaPipe + HaMeR local) leyendo desde `/content/` vs desde Drive.

**Requisito:** tener en Google Drive:
- `hamer_ckpt/_DATA/hamer_ckpts/checkpoints/hamer.safetensors`
- `hamer_ckpt/_DATA/data/` (archivos MANO)

In [ ]:
# Celda 1: Instalar dependencias
!pip install -q fastapi uvicorn nest-asyncio
!pip install -q git+https://github.com/Schmetzler/hamer_keypoints.git
!pip install -q "mediapipe==0.10.13"

In [ ]:
# Celda 2: Montar Drive + cargar HaMeR
from google.colab import drive
drive.mount('/content/drive')

import torch
import inspect
inspect.getargspec = inspect.getfullargspec
import hamer.configs as hcfg

HAMER_DATA_PATH = "/content/drive/MyDrive/hamer_ckpt/_DATA"
hcfg.CACHE_DIR_HAMER = HAMER_DATA_PATH

from hamer.hamer_module import HAMER

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

hamer_model = HAMER(mode="IMAGE", device=device)

import cv2, numpy as np
dummy = np.zeros((224, 224, 3), dtype=np.uint8)
hamer_model.process(dummy, {"Right": [0.0, 0.0, 1.0, 1.0]}, rescale_factor=1.0)
print("HaMeR cargado y warmup OK")

In [ ]:
# Celda 3: Descargar zip S2 y extraer
import os, subprocess, zipfile, random, shutil
from pathlib import Path

S2_URL = "https://www.dropbox.com/scl/fi/526mamb8nogb6gkbpmysv/GraspNet_S2_1_Source_data.zip?rlkey=cvygh7qev4amh52l01rz9ad1r&dl=1"
ZIP_PATH   = Path("/content/GraspNet_S2_1_Source_data.zip")
EXTRACT_LOCAL = Path("/content/benchmark_local")

if not ZIP_PATH.exists():
    print("Descargando zip S2 (~puede tardar varios minutos)...")
    subprocess.run(["wget", "-q", "--show-progress", "-O", str(ZIP_PATH), S2_URL], check=True)
    print("Descarga completada.")
else:
    print(f"ZIP ya existe: {ZIP_PATH}")

if not EXTRACT_LOCAL.exists():
    print("Extrayendo...")
    EXTRACT_LOCAL.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
        zf.extractall(EXTRACT_LOCAL)
    print("Extraccion completada.")
else:
    print(f"Ya extraido en: {EXTRACT_LOCAL}")

# Encontrar todos los jpgs
all_jpgs = sorted(EXTRACT_LOCAL.rglob("*.jpg"))
print(f"Total de jpgs encontrados: {len(all_jpgs):,}")

In [ ]:
# Celda 4: Seleccionar muestra de 200 frames
N_SAMPLE = 200
random.seed(42)
sample_paths_local = random.sample(all_jpgs, min(N_SAMPLE, len(all_jpgs)))
print(f"Muestra seleccionada: {len(sample_paths_local)} frames")
print("Ejemplo:", sample_paths_local[0])

In [ ]:
# Celda 5: Pipeline MediaPipe + HaMeR (funcion reutilizable)
import mediapipe as mp
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision as mp_vision
import time, urllib.request, os

CROP_SIZE     = 256
PADDING       = 0.3
WRIST_IDX     = 0
INDEX_MCP_IDX = 5
TARGET_DIST   = 0.1

# Descargar modelo de MediaPipe si no existe
MODEL_PATH = "/content/hand_landmarker.task"
if not os.path.exists(MODEL_PATH):
    urllib.request.urlretrieve(
        "https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/latest/hand_landmarker.task",
        MODEL_PATH
    )

base_options = mp_python.BaseOptions(model_asset_path=MODEL_PATH)
options = mp_vision.HandLandmarkerOptions(
    base_options=base_options,
    num_hands=1,
    min_hand_detection_confidence=0.3,
    min_hand_presence_confidence=0.3,
    min_tracking_confidence=0.3,
    running_mode=mp_vision.RunningMode.IMAGE,
)
detector = mp_vision.HandLandmarker.create_from_options(options)

def normalize(pts):
    pts = pts.copy()
    pts -= pts[WRIST_IDX]
    d = np.linalg.norm(pts[INDEX_MCP_IDX])
    if d > 1e-6:
        pts *= TARGET_DIST / d
    return pts

def run_pipeline(img_path):
    frame = cv2.imread(str(img_path))
    if frame is None:
        return None, None, "read_error"

    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
    result = detector.detect(mp_image)

    if not result.hand_landmarks:
        return None, None, "no_hand"

    # Handedness: MediaPipe reporta desde perspectiva de camara (no selfie)
    if result.handedness:
        label = result.handedness[0][0].category_name  # "Left" o "Right"
        is_right = (label == "Right")
    else:
        is_right = True

    # Bbox desde landmarks
    lms = result.hand_landmarks[0]
    xs = [lm.x for lm in lms]
    ys = [lm.y for lm in lms]
    h, w = frame.shape[:2]
    x1 = max(0.0, min(xs) - PADDING)
    y1 = max(0.0, min(ys) - PADDING)
    x2 = min(1.0, max(xs) + PADDING)
    y2 = min(1.0, max(ys) + PADDING)
    crop = frame[int(y1*h):int(y2*h), int(x1*w):int(x2*w)]
    if crop.size == 0:
        return None, None, "crop_empty"
    crop = cv2.resize(crop, (CROP_SIZE, CROP_SIZE))

    hand_side = "Right" if is_right else "Left"
    keypoints, _ = hamer_model.process(crop, {hand_side: [0.0, 0.0, 1.0, 1.0]}, rescale_factor=1.0)

    if hand_side not in keypoints:
        return None, None, "hamer_fail"

    kp = np.array(keypoints[hand_side], dtype=np.float32)

    try:
        hand_pose_rotmat = hamer_model.result["pred_mano_hand_pose"][0].numpy()
        aa_list = []
        for rot in hand_pose_rotmat:
            aa, _ = cv2.Rodrigues(rot)
            aa_list.extend(aa.flatten().tolist())
        mano_pose_aa = aa_list
    except Exception:
        return None, None, "pose_fail"

    kp_norm = normalize(kp)
    if not is_right:
        kp_norm[:, 0] *= -1.0

    return kp_norm, mano_pose_aa, None

print("Pipeline definido.")

In [ ]:
# Celda 6: Benchmark LOCAL
print(f"=== Benchmark LOCAL ({len(sample_paths_local)} frames desde /content/) ===")
n_ok_local = n_fail_local = 0
t0 = time.time()

for i, p in enumerate(sample_paths_local):
    kp, pose, err = run_pipeline(p)
    if kp is not None:
        n_ok_local += 1
    else:
        n_fail_local += 1
    if (i + 1) % 50 == 0:
        elapsed = time.time() - t0
        fps = (i + 1) / elapsed
        print(f"  [{i+1}/{len(sample_paths_local)}] {fps:.2f} fps | ok={n_ok_local} fail={n_fail_local}")

elapsed_local = time.time() - t0
fps_local = len(sample_paths_local) / elapsed_local
print(f"\nLOCAL: {fps_local:.2f} fps | ok={n_ok_local} fail={n_fail_local} | total={elapsed_local:.1f}s")

In [ ]:
# Celda 7: Copiar muestra a Drive para benchmark Drive
DRIVE_BENCHMARK = Path("/content/drive/MyDrive/hograspnet_benchmark")
DRIVE_BENCHMARK.mkdir(parents=True, exist_ok=True)

sample_paths_drive = []
print(f"Copiando {len(sample_paths_local)} frames a Drive...")
t0 = time.time()
for p in sample_paths_local:
    dst = DRIVE_BENCHMARK / p.name
    if not dst.exists():
        shutil.copy2(p, dst)
    sample_paths_drive.append(dst)
elapsed_copy = time.time() - t0
print(f"Copia completada en {elapsed_copy:.1f}s ({len(sample_paths_drive)} archivos)")

In [ ]:
# Celda 8: Benchmark DRIVE
print(f"=== Benchmark DRIVE ({len(sample_paths_drive)} frames desde Drive) ===")
n_ok_drive = n_fail_drive = 0
t0 = time.time()

for i, p in enumerate(sample_paths_drive):
    kp, pose, err = run_pipeline(p)
    if kp is not None:
        n_ok_drive += 1
    else:
        n_fail_drive += 1
    if (i + 1) % 50 == 0:
        elapsed = time.time() - t0
        fps = (i + 1) / elapsed
        print(f"  [{i+1}/{len(sample_paths_drive)}] {fps:.2f} fps | ok={n_ok_drive} fail={n_fail_drive}")

elapsed_drive = time.time() - t0
fps_drive = len(sample_paths_drive) / elapsed_drive
print(f"\nDRIVE: {fps_drive:.2f} fps | ok={n_ok_drive} fail={n_fail_drive} | total={elapsed_drive:.1f}s")

In [ ]:
# Celda 9: Resumen y estimación de tiempo total

# S1 ya procesado: ~5,453 frames
# S2-S99 = 97 sujetos restantes
# Estimamos misma cantidad que S1 por sujeto (conservador)
FRAMES_PER_SUBJECT_EST = 5_500
N_REMAINING_SUBJECTS   = 97
TOTAL_FRAMES_EST       = FRAMES_PER_SUBJECT_EST * N_REMAINING_SUBJECTS

def eta_str(fps, total_frames):
    if fps <= 0:
        return "N/A"
    secs = total_frames / fps
    h = int(secs // 3600)
    m = int((secs % 3600) // 60)
    return f"{h}h {m}min"

print("=" * 50)
print("RESULTADOS DEL BENCHMARK")
print("=" * 50)
print(f"  LOCAL  : {fps_local:.2f} fps")
print(f"  DRIVE  : {fps_drive:.2f} fps")
print(f"  Ratio  : {fps_local / max(fps_drive, 1e-9):.1f}x mas rapido el local")
print()
print(f"Estimacion para {N_REMAINING_SUBJECTS} sujetos (~{TOTAL_FRAMES_EST:,} frames):")
print(f"  Con LOCAL  : {eta_str(fps_local, TOTAL_FRAMES_EST)}")
print(f"  Con DRIVE  : {eta_str(fps_drive, TOTAL_FRAMES_EST)}")
print()
print("NOTA: estos tiempos son solo de inferencia.")
print("Sumar ~5-10 min por sujeto de descarga + extraccion del zip.")
print("Cada sesion de Colab T4 dura hasta ~12h.")
print()
print("Recomendacion: usar DRIVE para leer frames solo si fps_local < 2x fps_drive.")
print("En la practica: descargar zip a /content/, extraer a /content/, leer local, guardar CSV en Drive.")

In [ ]:
# Celda 10: Limpieza (opcional, ejecutar solo si quieres liberar espacio)
# import shutil
# shutil.rmtree(EXTRACT_LOCAL)
# ZIP_PATH.unlink(missing_ok=True)
# shutil.rmtree(DRIVE_BENCHMARK)
# print("Limpieza completada.")